# Bước 2.1: làm sạch kỹ thuật, trước → sau

**Mục đích:** xem bước làm sạch (Level 1) đã sửa những gì trong `text` để ra `text_clean`: bao nhiêu bình luận mỗi bước sửa, ví dụ cụ thể trước → sau. Luật: `docs/2-tien-xu-ly/dac-ta-lam-sach.md`. Code: `src/processing/clean.py`. Cập nhật lần cuối: 2026-11-19.

**Nội dung**

1. Đọc file đã xử lý
2. Số bình luận mỗi bước đã sửa (toàn bộ dữ liệu, từ manifest)
3. Từng bước trước → sau, và kiểm tra file khớp code
4. Danh sách bình luận bị sửa

**Đọc file nào:** `data/processed/comments.jsonl` và manifest của nó, do lệnh tầng 2 ghi ra. Chưa có file thì chạy trước, từ gốc repo: `.\.venv\Scripts\python.exe -m src.processing.run` (hướng dẫn tầng 2 mục 3). Notebook không ghi file nào.

In [ ]:
import json
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from helpers import *   # ROOT, đường dẫn dữ liệu, N_SHOW, doc_jsonl, doc_da_xu_ly, doc_bo_may, lo_ra, bao_cao, display, pd
thiet_lap_hien_thi()

from src.processing import PIPELINE_VERSION, chuan_hoa, tach_emoji, tach_tu
from src.processing import clauses as L3   # 2.3 tách mệnh đề
from src.processing import clean as L1     # 2.1 làm sạch
from src.processing import normalize as L2   # 2.2 chuẩn hóa

## 1. Đọc file đã xử lý

In [ ]:
df, man = doc_da_xu_ly(["comment_id", "video_id", "model", "text", "text_clean"])

## 2. Số bình luận mỗi bước đã sửa

**Đọc kết quả:** số của **toàn bộ** dữ liệu, lệnh tầng 2 đếm lúc chạy (manifest). Một bình luận có thể bị sửa ở nhiều bước. B1 là bình luận rỗng từ đầu, B7 là bình luận chỉ có ký tự vô hình, thành rỗng sau làm sạch.

In [ ]:
buoc = {k: v for k, v in man["so_binh_luan_moi_buoc_sua"].items() if k.startswith("B")}
bang_buoc = pd.Series(buoc, name="số bình luận bị sửa").to_frame()
bang_buoc["tỷ lệ (%)"] = (100 * bang_buoc["số bình luận bị sửa"] / man["ra"]["so_dong"]).round(3)
display(bang_buoc)

## 3. Từng bước trước → sau

**Để làm gì:** sửa các lỗi **kỹ thuật** trong chữ: lỗi do cách máy lưu và truyền văn bản, không phải do người viết. Level 1 **không đổi nghĩa** bình luận:
- không đổi chữ hoa/thường;
- không sửa teencode, chính tả;
- không bỏ emoji;
- không lọc spam.

Các việc đó thuộc level sau.

| Bước | Lỗi | Cách xử lý |
|---|---|---|
| B1 | Comment null | Đánh dấu bình luận trống (null, rỗng, chỉ có khoảng trắng) |
| B2 | HTML entities | `&#39;` → `'`, `&amp;` → `&`, `&quot;` → `"` … |
| B3 | Ký tự điều khiển không cần thiết | Bỏ ký tự điều khiển và ký tự vô hình (zero-width space, BOM, word joiner, NUL…) |
| B4 | Unicode không thống nhất | Chuẩn NFC: chữ có dấu ghép từ nhiều ký tự gộp về một ký tự dựng sẵn |
| B5 | Xuống dòng bất thường | `\r\n`, `\r` và các ký tự xuống dòng lạ → `\n`; nhiều dòng trống liền nhau → một lần xuống dòng |
| B6 | Khoảng trắng dư | Tab, khoảng trắng lạ (NBSP…) → dấu cách; nhiều dấu cách → một; bỏ khoảng trắng đầu/cuối |
| B7 | Comment null sau làm sạch | Đánh dấu bình luận chỉ chứa ký tự vô hình, sau B3–B6 thành rỗng |

**Vì sao theo thứ tự này:**
- B2 chạy trước vì giải mã entity có thể sinh ra ký tự điều khiển hay khoảng trắng lạ (vd `&nbsp;`), các bước sau sẽ dọn.
- B3 bỏ ký tự vô hình trước B4, vì ký tự vô hình chen giữa chữ và dấu làm NFC không gộp được.
- B5 chạy trước B6, để dòng chỉ có dấu cách cũng được tính là dòng trống.

**Chạy lại trên nhóm nào:** chỉ những bình luận mà file ghi `text_clean` khác `text`. Bình luận không bị sửa thì bước nào cũng giữ nguyên, nên số đếm ở đây vẫn là của toàn bộ dữ liệu.

**Mỗi ô code bên dưới:** gọi hàm sửa một loại lỗi trong `src/processing/clean.py` (lệnh chạy chuẩn hóa ra file cũng dùng đúng các hàm này), áp lên toàn bộ bình luận, rồi in:
- số bình luận bị sửa;
- tối đa `N_SHOW` ví dụ trước → sau.

### B1. Comment null

**Để làm gì:** tìm bình luận không có nội dung: `text` là null, là chuỗi rỗng, hoặc chỉ có khoảng trắng. Không có chữ thì không phân tích được.

**Ô code làm gì:**
- Đếm số bình luận này. Trong file đã xử lý, chúng có `text_clean` rỗng.
- In danh sách bình luận null.

Bình luận null **không bị xóa** khỏi bảng, chỉ được đánh dấu. Bỏ hay giữ là việc của bước sau.

In [ ]:
la_null = df["text"].map(L1.la_rong)
print(f"Comment null: {la_null.sum():,} / {len(df):,}")
display(df.loc[la_null, ["comment_id", "video_id", "model"]].assign(text=df.loc[la_null, "text"].map(lo_ra)).head(N_SHOW))

xet = df[~la_null & (df["text"] != df["text_clean"])]   # bình luận có nội dung mà Level 1 có sửa
text = xet["text"].astype(str)                          # chuỗi đang làm sạch, đi qua lần lượt B2 → B6
BAO_CAO = []
print(f"Bình luận có nội dung bị Level 1 sửa: {len(xet):,}. Các bước dưới chạy lại trên nhóm này; % tính trên toàn bộ {len(df):,} bình luận.")

### B2. HTML entities

**Để làm gì:** HTML entity là cách viết ký tự đặc biệt trong trang web: `&#39;` là `'`, `&amp;` là `&`, `&quot;` là `"`, `&lt;` là `<`. Bình luận YouTube đôi khi còn sót dạng này. Không đổi lại thì cùng một câu có hai cách viết, và `&#39;` bị coi như chữ.

**Ô code làm gì:**
- Chỉ đổi entity viết đúng dạng `&tên;`, `&#số;`, `&#xHEX;` (phải có dấu `;`), để không đụng vào chữ thường như "AT&T".
- Lặp tối đa 3 lần cho chuỗi bị mã hóa chồng, vd `&amp;#39;` → `&#39;` → `'`.

In [ ]:
truoc, text = text, text.map(L1.giai_ma_html)
bao_cao("B2. HTML entities", truoc, text, ma=xet["comment_id"], tong=len(df), ghi_vao=BAO_CAO)

### B3. Ký tự điều khiển không cần thiết

**Để làm gì:** có những ký tự không hiện ra màn hình nhưng vẫn nằm trong chuỗi. Chúng làm hai câu nhìn giống hệt nhau mà máy coi là khác nhau, và có thể làm vỡ việc tách từ. Ví dụ:
- zero-width space `U+200B`: hay dính sau tên người được nhắc;
- word joiner `U+2060`;
- BOM `U+FEFF`;
- ký tự đổi chiều chữ `U+200E`;
- ký tự điều khiển cũ như NUL `U+0000`, backspace `U+0008`, DEL `U+007F`.

**Ô code làm gì:**
- Bỏ mọi ký tự thuộc nhóm Unicode `Cc` (điều khiển) và `Cf` (định dạng, vô hình), cùng các mảnh surrogate lỗi.
- **Giữ lại**, vì là một phần của nội dung:
  - xuống dòng, tab, `\r`: để B5, B6 xử lý;
  - zero-width joiner `U+200D` khi nằm **giữa hai emoji**: nó ghép emoji, vd 👨‍👩‍👧 hay 🏳️‍🌈. ZWJ đứng lẻ hoặc cạnh chữ thì vẫn bỏ;
  - ký tự tag `U+E0020–E007F`: thuộc emoji cờ như 🏴󠁧󠁢󠁥󠁮󠁧󠁿.
- Ký tự riêng (Private Use, vd logo Apple `U+F8FF`) và emoji mới mà Python chưa biết **không** bị bỏ: chúng không phải ký tự điều khiển.

In [ ]:
truoc, text = text, text.map(L1.bo_ky_tu_dieu_khien)
bao_cao("B3. Ký tự điều khiển, ký tự vô hình", truoc, text, ma=xet["comment_id"], tong=len(df), ghi_vao=BAO_CAO)

### B4. Unicode không thống nhất

**Để làm gì:** một chữ tiếng Việt có dấu có thể được lưu theo hai cách, nhìn giống hệt nhau:
- **dựng sẵn:** `à` là 1 ký tự (`U+00E0`);
- **tổ hợp:** `a` + dấu huyền rời (`U+0300`), tức 2 ký tự.

Máy coi hai cách là hai chữ khác nhau. Không chuẩn hóa thì tìm "màn hình" sẽ trượt những bình luận gõ kiểu tổ hợp, và đếm từ bị tách đôi.

**Ô code làm gì:** chuẩn hóa **NFC**, đưa mọi chữ về dạng dựng sẵn. NFC không đổi nội dung, chỉ đổi cách lưu.

**Không làm ở đây:**
- **NFKC:** nó còn đổi cả nội dung như `①` → `1`, chữ kiểu cách `𝐀` → `A`, `…` → `...`.
- **Thống nhất kiểu bỏ dấu cũ/mới** (`hoà`/`hòa`, `thuý`/`thúy`): đây là chuẩn hóa ngôn ngữ, không phải lỗi kỹ thuật.

In [ ]:
truoc, text = text, text.map(L1.chuan_unicode)
bao_cao("B4. Unicode NFC", truoc, text, ma=xet["comment_id"], tong=len(df), ghi_vao=BAO_CAO)

### B5. Xuống dòng bất thường

**Để làm gì:** cùng là "xuống dòng" nhưng máy có nhiều cách ghi:
- `\r\n` (Windows), `\r` (máy Mac cũ);
- `U+0085`.

`U+2028`, `U+2029` không đổi ở bước này: Python coi chúng là khoảng trắng, nên B6 đổi thành dấu cách.

Ngoài ra nhiều bình luận có cả chục dòng trống liền nhau.

**Ô code làm gì:**
1. Đổi mọi kiểu xuống dòng về `\n`.
2. Gộp nhiều dòng trống liền nhau, kể cả dòng chỉ có dấu cách, thành **một** lần xuống dòng.

**Không** đổi xuống dòng thành dấu cách: người viết thường xuống dòng để tách ý, bước tách mệnh đề sau này cần thông tin đó. Xuống dòng ở đầu/cuối bình luận được bỏ ở B6.

In [ ]:
truoc, text = text, text.map(L1.chuan_xuong_dong)
bao_cao("B5. Xuống dòng", truoc, text, ma=xet["comment_id"], tong=len(df), ghi_vao=BAO_CAO)

### B6. Khoảng trắng dư

**Để làm gì:** khoảng trắng thừa không mang nghĩa nhưng làm cùng một câu thành nhiều chuỗi khác nhau. Gồm:
- nhiều dấu cách liền nhau;
- khoảng trắng ở đầu/cuối bình luận, đầu/cuối mỗi dòng;
- tab;
- các khoảng trắng "lạ" nhìn như dấu cách, vd khoảng trắng không ngắt dòng NBSP `U+00A0`, khoảng trắng chữ Nhật `U+3000`.

**Ô code làm gì:**
1. Đổi mọi khoảng trắng (trừ xuống dòng) thành dấu cách thường, gộp nhiều cái liền nhau thành một.
2. Bỏ dấu cách sát hai bên chỗ xuống dòng.
3. Bỏ khoảng trắng và xuống dòng ở đầu, cuối bình luận.

In [ ]:
truoc, text = text, text.map(L1.chuan_khoang_trang)
bao_cao("B6. Khoảng trắng", truoc, text, ma=xet["comment_id"], tong=len(df), ghi_vao=BAO_CAO)

### B7. Comment null sau làm sạch

**Để làm gì:** có bình luận nhìn thì có nội dung nhưng thực ra chỉ chứa ký tự vô hình, vd một chuỗi zero-width joiner. B1 không bắt được loại này. Sau B3–B6, chúng thành chuỗi rỗng.

**Ô code làm gì:**
- Cộng chung với B1 ở bảng tổng kết.
- In ra kèm nội dung gốc, ký tự vô hình hiện thành mã.

In [ ]:
null_sau = text == ""
idx_null_sau = null_sau[null_sau].index
print(f"Thành rỗng sau làm sạch: {len(idx_null_sau):,}")
display(xet.loc[idx_null_sau, ["comment_id", "video_id", "model"]].assign(text_goc=xet.loc[idx_null_sau, "text"].map(lo_ra)).head(N_SHOW))
so_null = int(la_null.sum()) + len(idx_null_sau)
BAO_CAO.append({"bước": "B1 + B7. Comment null (tổng)", "số bình luận bị sửa": so_null, "tỷ lệ (%)": round(100 * so_null / len(df), 3)})

### Tổng kết Level 1

**Đọc kết quả:**
- Mỗi dòng là một bước và số bình luận bước đó đã sửa. Một bình luận có thể bị sửa ở nhiều bước.
- Dòng cuối là tổng số comment null (B1 + B7).
- Dòng "Có thay đổi" là số bình luận mà `text_clean` khác `text` gốc.
- Ô còn so kết quả chạy lại các bước với `text_clean` trong file. Khác nhau nghĩa là file được tạo bằng code cũ: chạy lại lệnh tầng 2.

In [ ]:
co_doi = df["text"].fillna("") != df["text_clean"]
display(pd.DataFrame(BAO_CAO + [{"bước": "Có thay đổi sau Level 1", "số bình luận bị sửa": int(co_doi.sum()),
                                 "tỷ lệ (%)": round(100 * co_doi.mean(), 3)}]))
lech = text != xet["text_clean"]
if lech.any():
    print(f"⚠ {lech.sum():,} bình luận: chạy lại các bước cho kết quả khác text_clean trong file. Chạy lại lệnh tầng 2.")
else:
    print("Chạy lại các bước cho kết quả giống hệt text_clean trong file.")

## 4. Danh sách bình luận bị sửa

**Ô code làm gì:** in `N_SHOW` bình luận đầu tiên bị làm sạch sửa (bỏ bình luận rỗng), đặt `text` gốc cạnh `text_clean`. Ký tự vô hình hiện thành mã, xuống dòng hiện là `⏎`, dấu cách thừa hiện là `·`, để thấy đã bỏ gì.

In [ ]:
bi_sua = df.loc[co_doi & (df["text_clean"] != ""), ["comment_id", "model", "text", "text_clean"]].head(N_SHOW)
display(bi_sua.assign(text=bi_sua["text"].map(lo_ra), text_clean=bi_sua["text_clean"].map(lo_ra)))